In [4]:
import pandas as pd

In [ ]:
game_infos = pd.read_csv("archive/Games.csv")

In [ ]:
games = pd.read_csv("archive/TeamStatistics.csv")

In [ ]:
games.columns

In [ ]:
all_games = transform_nba_dataframe(games, game_infos)

In [ ]:
print(date_to_num("2025-04-01 20:00:00"))

In [72]:
from datetime import datetime

def date_to_num(date_str: str) -> int:
    """
    Convert a datetime string to a unique integer for that calendar day.
    Example: "2025-06-22 20:00:00" -> 739372
    """
    # Parse string into datetime
    dt = datetime.strptime(date_str, "%Y-%m-%d %H:%M:%S")
    
    # Convert to unique day number
    return dt.date().toordinal()


In [ ]:
import pandas as pd
import numpy as np

def transform_nba_dataframe(df, game_infos):
    # Create a dictionary to store processed game IDs
    processed_games = {}
    new_df_list = []
    
    for _, row in df.iterrows():
        game_id = row['gameId']
        
        # Skip if we've already processed this game
        if game_id in processed_games:
            continue
            
        processed_games[game_id] = True
        
        # Find the matching row for this game (the other team's perspective)
        matching_rows = df[(df['gameId'] == game_id) & 
                          (df['teamId'] != row['teamId'])]
        matching_game = game_infos[(game_infos['gameId'] == game_id)]
        
        if len(matching_rows) == 0:
            continue  # Skip if no matching row found

        if len(matching_game) == 0:
            continue  # Skip if no matching row found

        opponent_row = matching_rows.iloc[0]
        game_row = matching_game.iloc[0]
        
        # Determine home and away teams
        if row['home']:
            home_team_row = row
            visitor_team_row = opponent_row
        else:
            home_team_row = opponent_row
            visitor_team_row = row

        visitor_name = visitor_team_row['teamCity'] + ' ' + visitor_team_row['teamName']
        home_name = home_team_row['teamCity'] + ' ' + home_team_row['teamName']
        # Create a new row for the transformed dataframe
        new_row = {
            'Game ID': home_team_row['gameId'],
            'Date Number': date_to_num(home_team_row['gameDate']),
            'Date': home_team_row['gameDate'],
            'Game Type': game_row['gameType'],
            'Home Team': home_name,
            'Visitor Team': visitor_name,
            'Winner': home_name if home_team_row['win'] else visitor_name,
            'Loser': visitor_name if home_team_row['win'] else home_name,
            'Home Win': home_team_row['win'],
            'Home Team Total Games':  home_team_row['seasonWins'] + home_team_row['seasonLosses'],
            'Visitor Team Total Games':  visitor_team_row['seasonWins'] + visitor_team_row['seasonLosses'],
            'Home Team W':  home_team_row['seasonWins'],
            'Home Team L':  home_team_row['seasonLosses'],
            'Visitor Team W':  visitor_team_row['seasonWins'],
            'Visitor Team L':  visitor_team_row['seasonLosses'],
            
            # Home team stats
            'Home Team Points': home_team_row['teamScore'],
            'Home Team Assists': home_team_row['assists'],
            'Home Team Tot Rebounds': home_team_row['reboundsTotal'],
            'Home Team Off Rebounds': home_team_row['reboundsOffensive'],
            'Home Team Def Rebounds': home_team_row['reboundsDefensive'],
            'Home Team Blocks': home_team_row['blocks'],
            'Home Team Steals': home_team_row['steals'],
            'Home Team FGA': home_team_row['fieldGoalsAttempted'],
            'Home Team FGM': home_team_row['fieldGoalsMade'],
            'Home Team FG%': home_team_row['fieldGoalsPercentage'],
            'Home Team 3PA': home_team_row['threePointersAttempted'],
            'Home Team 3PM': home_team_row['threePointersMade'],
            'Home Team 3P%': home_team_row['threePointersPercentage'],
            'Home Team FTA': home_team_row['freeThrowsAttempted'],
            'Home Team FTM': home_team_row['freeThrowsMade'],
            'Home Team FT%': home_team_row['freeThrowsPercentage'],
            'Home Team Fouls': home_team_row['foulsPersonal'],
            'Home Team TO': home_team_row['turnovers'],
            
            # Visitor team stats
            'Visitor Team Points': visitor_team_row['teamScore'],
            'Visitor Team Assists': visitor_team_row['assists'],
            'Visitor Team Tot Rebounds': visitor_team_row['reboundsTotal'],
            'Visitor Team Off Rebounds': visitor_team_row['reboundsOffensive'],
            'Visitor Team Def Rebounds': visitor_team_row['reboundsDefensive'],
            'Visitor Team Blocks': visitor_team_row['blocks'],
            'Visitor Team Steals': visitor_team_row['steals'],
            'Visitor Team FGA': visitor_team_row['fieldGoalsAttempted'],
            'Visitor Team FGM': visitor_team_row['fieldGoalsMade'],
            'Visitor Team FG%': visitor_team_row['fieldGoalsPercentage'],
            'Visitor Team 3PA': visitor_team_row['threePointersAttempted'],
            'Visitor Team 3PM': visitor_team_row['threePointersMade'],
            'Visitor Team 3P%': visitor_team_row['threePointersPercentage'],
            'Visitor Team FTA': visitor_team_row['freeThrowsAttempted'],
            'Visitor Team FTM': visitor_team_row['freeThrowsMade'],
            'Visitor Team FT%': visitor_team_row['freeThrowsPercentage'],
            'Visitor Team Fouls': visitor_team_row['foulsPersonal'],
            'Visitor Team TO': visitor_team_row['turnovers']
        }
        
        new_df_list.append(new_row)
    
    # Create the new DataFrame
    new_df = pd.DataFrame(new_df_list)
    new_df = new_df.sort_values(by="Date Number").reset_index(drop=True)
    new_df = new_df[new_df["Game Type"] != 'Preseason']
    new_df["Playoff Game"] = ((new_df["Game Type"] == "Playoffs") | (new_df["Game Type"] == "Play-in Tournament")).astype(int)
    # Ensure columns are in the requested order
    # column_order = ['Date Number', 'Date',
    #     'Visitor Team', 'Visitor Team Points', 'Home Team', 'Home Team Points', 
    #     'Winner', 'Loser', 'Home Win', 'Home Team Assists', 'Home Team Tot Rebounds', 
    #     'Home Team Off Rebounds', 'Home Team Def Rebounds', 'Home Team Blocks', 
    #     'Home Team Steals', 'Home Team FGA', 'Home Team FGM', 'Home Team FG%',
    #     'Home Team 3PA', 'Home Team 3PM', 'Home Team 3P%', 'Home Team FTA', 
    #     'Home Team FTM', 'Home Team FT%', 'Home Team Fouls', 'Home Team TO', 
    #     'Visitor Team Assists', 'Visitor Team Tot Rebounds', 'Visitor Team Off Rebounds', 
    #     'Visitor Team Def Rebounds', 'Visitor Team Blocks', 'Visitor Team Steals',
    #     'Visitor Team FGA', 'Visitor Team FGM', 'Visitor Team FG%', 'Visitor Team 3PA', 
    #     'Visitor Team 3PM', 'Visitor Team 3P%', 'Visitor Team FTA', 'Visitor Team FTM', 
    #     'Visitor Team FT%', 'Visitor Team Fouls', 'Visitor Team TO'
    # ]
    
    return new_df

# Example usage:
# transformed_df = transform_nba_dataframe_deduplicated(original_df)

In [ ]:
all_games.to_csv("csvs/all_games.csv")

In [ ]:
all_games = all_games.sort_values(by="Date Number").reset_index(drop=True)

In [ ]:
all_games["Game Type"].unique()

In [ ]:
all_games = all_games[all_games["Game Type"] != 'Preseason']

In [ ]:
all_games["Playoff Game"] = ((all_games["Game Type"] == "Playoffs") | (all_games["Game Type"] == "Play-in Tournament")).astype(int)

In [ ]:
all_games.columns

In [ ]:
season_id = (all_games["Game Type"].eq("Regular Season") & 
             all_games["Game Type"].shift().eq("Playoffs")).cumsum()
all_games["Season"] = season_id + 1947


In [ ]:
import requests
for year in years:
    url = url_start.format(year)
    data = requests.get(url)
    
    with open("mvps/{}.html".format(year), "w+") as f:
        f.write(data.text)

In [ ]:
import pandas as pd
import io

# Your CSV string
csv_string = """Starters,MP,FG,FGA,FG%,3P,3PA,3P%,FT,FTA,FT%,ORB,DRB,TRB,AST,STL,BLK,TOV,PF,PTS,GmSc,+/-,-9999
Devin Vassell,33:30,6,13,.462,3,8,.375,2,2,1.000,0,8,8,2,0,1,2,2,17,12.0,-15,vassede01
Stephon Castle,27:33,8,19,.421,3,6,.500,4,6,.667,2,6,8,3,0,0,2,5,23,13.4,-8,castlst01
Chris Paul,22:05,3,7,.429,2,3,.667,1,1,1.000,1,5,6,8,0,0,2,0,9,11.1,-12,paulch01
Bismack Biyombo,19:51,2,3,.667,0,0,,0,0,,0,4,4,1,0,0,0,3,4,3.4,-7,biyombi01
Harrison Barnes,19:18,3,6,.500,1,2,.500,0,0,,1,1,2,3,0,0,1,0,7,6.1,-13,barneha02
Julian Champagnie,28:31,2,8,.250,2,7,.286,0,0,,0,4,4,5,1,1,1,1,6,6.2,-4,champju02
Keldon Johnson,27:59,4,10,.400,0,2,.000,3,3,1.000,1,3,4,2,2,2,3,2,11,8.2,-7,johnske04
Jeremy Sochan,26:06,7,9,.778,1,3,.333,0,0,,2,1,3,1,0,0,3,3,15,9.7,-11,sochaje01
Blake Wesley,22:21,3,7,.429,1,3,.333,0,0,,0,3,3,5,2,1,2,1,7,8.0,-3,weslebl01
Sandro Mamukelashvili,12:46,3,6,.500,2,4,.500,2,2,1.000,2,0,2,0,1,0,1,1,10,8.0,0,mamuksa01
Jordan McLaughlin,Did Not Play,Did Not Play,Did Not Play,Did Not Play,Did Not Play,Did Not Play,Did Not Play,Did Not Play,Did Not Play,Did Not Play,Did Not Play,Did Not Play,Did Not Play,Did Not Play,Did Not Play,Did Not Play,Did Not Play,Did Not Play,Did Not Play,Did Not Play,Did Not Play,mclaujo01
Team Totals,240,41,88,.466,15,38,.395,12,14,.857,9,35,44,30,6,5,17,18,109,,,-9999"""

csv_string2 = """Starters,MP,FG,FGA,FG%,3P,3PA,3P%,FT,FTA,FT%,ORB,DRB,TRB,AST,STL,BLK,TOV,PF,PTS,GmSc,+/-,-9999
Austin Reaves,36:10,12,21,.571,5,13,.385,1,1,1.000,0,7,7,6,2,0,2,0,30,26.4,+18,reaveau01
Luka Dončić,33:56,5,20,.250,1,7,.143,10,13,.769,2,7,9,14,3,1,3,1,21,21.4,+19,doncilu01
Jordan Goodwin,32:40,6,9,.667,3,6,.500,0,0,,1,3,4,3,3,1,0,2,15,17.7,+9,goodwjo01
Dorian Finney-Smith,27:59,5,12,.417,4,8,.500,1,2,.500,2,1,3,1,0,0,0,1,15,10.2,+13,finnedo01
Jaxson Hayes,27:36,3,5,.600,0,0,,3,4,.750,2,9,11,3,2,1,2,1,9,12.8,+21,hayesja02
Gabe Vincent,27:58,4,6,.667,3,5,.600,0,0,,0,2,2,1,2,0,0,3,11,10.5,+9,vincega01
Dalton Knecht,20:24,4,10,.400,2,7,.286,3,4,.750,1,2,3,3,0,1,0,1,13,10.9,-5,knechda01
Jarred Vanderbilt,14:54,1,2,.500,0,0,,0,0,,0,1,1,1,0,0,1,1,2,0.6,-2,vandeja01
Christian Koloko,8:50,2,2,1.000,0,0,,0,0,,0,2,2,1,0,0,1,0,4,3.7,-4,kolokch01
Shake Milton,7:03,1,1,1.000,0,0,,0,0,,0,0,0,1,1,0,0,0,2,3.4,+4,miltosh01
Bronny James,2:30,1,2,.500,1,2,.500,0,0,,0,0,0,0,0,0,0,0,3,2.0,-2,jamesbr02
Alex Len,Did Not Play,Did Not Play,Did Not Play,Did Not Play,Did Not Play,Did Not Play,Did Not Play,Did Not Play,Did Not Play,Did Not Play,Did Not Play,Did Not Play,Did Not Play,Did Not Play,Did Not Play,Did Not Play,Did Not Play,Did Not Play,Did Not Play,Did Not Play,Did Not Play,lenal01
Markieff Morris,Did Not Play,Did Not Play,Did Not Play,Did Not Play,Did Not Play,Did Not Play,Did Not Play,Did Not Play,Did Not Play,Did Not Play,Did Not Play,Did Not Play,Did Not Play,Did Not Play,Did Not Play,Did Not Play,Did Not Play,Did Not Play,Did Not Play,Did Not Play,Did Not Play,morrima02
Cam Reddish,Did Not Play,Did Not Play,Did Not Play,Did Not Play,Did Not Play,Did Not Play,Did Not Play,Did Not Play,Did Not Play,Did Not Play,Did Not Play,Did Not Play,Did Not Play,Did Not Play,Did Not Play,Did Not Play,Did Not Play,Did Not Play,Did Not Play,Did Not Play,Did Not Play,reddica01
Team Totals,240,44,90,.489,19,48,.396,18,24,.750,8,34,42,34,13,4,9,10,125,,,-9999

"""
# Method 1: Using StringIO to read the CSV string
df = pd.read_csv(io.StringIO(csv_string))
df2 = pd.read_csv(io.StringIO(csv_string2))
# Save the DataFrame to a CSV file
df.to_csv('spurs(lakers).csv', index=False)
df2.to_csv('lakers(spurs).csv', index=False)

# # If you want to preserve the exact format without pandas interpreting the data
# with open('box_score_raw.csv', 'w') as f:
#     f.write(csv_string)

# print("CSV files have been saved successfully!")

In [ ]:
all_games.to_csv('all_games_2.csv')

In [ ]:
modern["Game ID"] = range(20905, 70083)

In [ ]:
all_games = all_games.loc[:, ~all_games.columns.str.contains("^Unnamed")]

In [ ]:
visitor_df = pd.read_csv('MissingGames/hornets(clippers).csv')
home_df = pd.read_csv('MissingGames/clippers(hornets).csv')
home_team_row = home_df[home_df["Starters"] == "Team Totals"].iloc[0]
visitor_team_row = visitor_df[visitor_df["Starters"] == "Team Totals"].iloc[0]
date = '2025-03-16 19:00:00'
home = 'Los Angeles Clippers'
visitor = "Charlotte Hornets"

new_row = {
            'Game ID': 0,
            'Date Number': date_to_num(date),
            'Date': date,
            'Game Type': 'Regular Season',
            'Home Team': home,
            'Visitor Team': visitor,
            'Winner': home,
            'Loser': visitor,
            'Home Win': 0,
            'Home Team Total Games': 0,
            'Visitor Team Total Games': 0,
            'Home Team W': 0,
            'Home Team L': 0,
            'Visitor Team W': 0,
            'Visitor Team L': 0,
            
            # Home team stats
            'Home Team Points': home_team_row['PTS'],
            'Home Team Assists': home_team_row['AST'],
            'Home Team Tot Rebounds': home_team_row['TRB'],
            'Home Team Off Rebounds': home_team_row['ORB'],
            'Home Team Def Rebounds': home_team_row['DRB'],
            'Home Team Blocks': home_team_row['BLK'],
            'Home Team Steals': home_team_row['STL'],
            'Home Team FGA': home_team_row['FGA'],
            'Home Team FGM': home_team_row['FG'],
            'Home Team FG%': home_team_row['FG%'],
            'Home Team 3PA': home_team_row['3PA'],
            'Home Team 3PM': home_team_row['3P'],
            'Home Team 3P%': home_team_row['3P%'],
            'Home Team FTA': home_team_row['FTA'],
            'Home Team FTM': home_team_row['FT'],
            'Home Team FT%': home_team_row['FT%'],
            'Home Team Fouls': home_team_row['PF'],
            'Home Team TO': home_team_row['TOV'],
            
            # Visitor team stats
            'Visitor Team Points': visitor_team_row['PTS'],
            'Visitor Team Assists': visitor_team_row['AST'],
            'Visitor Team Tot Rebounds': visitor_team_row['TRB'],
            'Visitor Team Off Rebounds': visitor_team_row['ORB'],
            'Visitor Team Def Rebounds': visitor_team_row['DRB'],
            'Visitor Team Blocks': visitor_team_row['BLK'],
            'Visitor Team Steals': visitor_team_row['STL'],
            'Visitor Team FGA': visitor_team_row['FGA'],
            'Visitor Team FGM': visitor_team_row['FG'],
            'Visitor Team FG%': visitor_team_row['FG%'],
            'Visitor Team 3PA': visitor_team_row['3PA'],
            'Visitor Team 3PM': visitor_team_row['3P'],
            'Visitor Team 3P%': visitor_team_row['3P%'],
            'Visitor Team FTA': visitor_team_row['FTA'],
            'Visitor Team FTM': visitor_team_row['FT'],
            'Visitor Team FT%': visitor_team_row['FT%'],
            'Visitor Team Fouls': visitor_team_row['PF'],
            'Visitor Team TO': visitor_team_row['TOV'],
            'Playoff Game': 0,
            'Season': 2025
        }
new_row_df = pd.DataFrame([new_row])
columns = ['Visitor Team Points', 'Home Team Points', 'Home Team Assists', 'Home Team Tot Rebounds', 
        'Home Team Off Rebounds', 'Home Team Def Rebounds', 'Home Team Blocks', 
        'Home Team Steals', 'Home Team FGA', 'Home Team FGM', 'Home Team FG%',
        'Home Team 3PA', 'Home Team 3PM', 'Home Team 3P%', 'Home Team FTA', 
        'Home Team FTM', 'Home Team FT%', 'Home Team Fouls', 'Home Team TO', 
        'Visitor Team Assists', 'Visitor Team Tot Rebounds', 'Visitor Team Off Rebounds', 
        'Visitor Team Def Rebounds', 'Visitor Team Blocks', 'Visitor Team Steals',
        'Visitor Team FGA', 'Visitor Team FGM', 'Visitor Team FG%', 'Visitor Team 3PA', 
        'Visitor Team 3PM', 'Visitor Team 3P%', 'Visitor Team FTA', 'Visitor Team FTM', 
        'Visitor Team FT%', 'Visitor Team Fouls', 'Visitor Team TO'
    ]
for col in columns:
    new_row_df[col] = new_row_df[col].astype(float)

# print(list(new_row_df.columns))
# print(list(all_games.columns))
# # Get the index of the row with Game ID 69787
# target_index = all_games.index[all_games["Game ID"] == 69888][0]

# # Replace it with the new row (ensure columns match)
# all_games.loc[target_index] = new_row_df.iloc[0].values

all_games = pd.concat([all_games, new_row_df], ignore_index=True)
all_games.to_csv('csvs/all_games.csv')

In [ ]:
print(list(modern.isna().sum()))

In [ ]:
columns = ['Visitor Team Points', 'Home Team Points', 'Home Team Assists', 'Home Team Tot Rebounds', 
        'Home Team Off Rebounds', 'Home Team Def Rebounds', 'Home Team Blocks', 
        'Home Team Steals', 'Home Team FGA', 'Home Team FGM', 'Home Team FG%',
        'Home Team 3PA', 'Home Team 3PM', 'Home Team 3P%', 'Home Team FTA', 
        'Home Team FTM', 'Home Team FT%', 'Home Team Fouls', 'Home Team TO', 
        'Visitor Team Assists', 'Visitor Team Tot Rebounds', 'Visitor Team Off Rebounds', 
        'Visitor Team Def Rebounds', 'Visitor Team Blocks', 'Visitor Team Steals',
        'Visitor Team FGA', 'Visitor Team FGM', 'Visitor Team FG%', 'Visitor Team 3PA', 
        'Visitor Team 3PM', 'Visitor Team 3P%', 'Visitor Team FTA', 'Visitor Team FTM', 
        'Visitor Team FT%', 'Visitor Team Fouls', 'Visitor Team TO'
    ]
for col in columns:
    all_games[col] = all_games[col].astype(float)

In [ ]:
all_games.to_csv("csvs/all_games.csv")

In [ ]:
all_games['Game ID'] = range(0, 70082)

In [ ]:
modern = all_games[all_games["Season"] >= 1986]

In [ ]:
print(list(modern.columns))

In [ ]:
cols = ['Home Team Points Avg', 'Visitor Team Points Avg', 'Home Team Opp Points Avg', 'Visitor Team Opp Points Avg', 'Home Team Assists Avg', 'Visitor Team Assists Avg', 'Home Team Opp Assists Avg', 'Visitor Team Opp Assists Avg', 'Home Team Tot Rebounds Avg', 'Visitor Team Tot Rebounds Avg', 'Home Team Opp Tot Rebounds Avg', 'Visitor Team Opp Tot Rebounds Avg', 'Home Team Off Rebounds Avg', 'Visitor Team Off Rebounds Avg', 'Home Team Opp Off Rebounds Avg', 'Visitor Team Opp Off Rebounds Avg', 'Home Team Def Rebounds Avg', 'Visitor Team Def Rebounds Avg', 'Home Team Opp Def Rebounds Avg', 'Visitor Team Opp Def Rebounds Avg', 'Home Team Blocks Avg', 'Visitor Team Blocks Avg', 'Home Team Opp Blocks Avg', 'Visitor Team Opp Blocks Avg', 'Home Team Steals Avg', 'Visitor Team Steals Avg', 'Home Team Opp Steals Avg', 'Visitor Team Opp Steals Avg', 'Home Team FGA Avg', 'Visitor Team FGA Avg', 'Home Team Opp FGA Avg', 'Visitor Team Opp FGA Avg', 'Home Team FGM Avg', 'Visitor Team FGM Avg', 'Home Team Opp FGM Avg', 'Visitor Team Opp FGM Avg', 'Home Team FG% Avg', 'Visitor Team FG% Avg', 'Home Team Opp FG% Avg', 'Visitor Team Opp FG% Avg', 'Home Team 3PA Avg', 'Visitor Team 3PA Avg', 'Home Team Opp 3PA Avg', 'Visitor Team Opp 3PA Avg', 'Home Team 3PM Avg', 'Visitor Team 3PM Avg', 'Home Team Opp 3PM Avg', 'Visitor Team Opp 3PM Avg', 'Home Team 3P% Avg', 'Visitor Team 3P% Avg', 'Home Team Opp 3P% Avg', 'Visitor Team Opp 3P% Avg', 'Home Team FTA Avg', 'Visitor Team FTA Avg', 'Home Team Opp FTA Avg', 'Visitor Team Opp FTA Avg', 'Home Team FTM Avg', 'Visitor Team FTM Avg', 'Home Team Opp FTM Avg', 'Visitor Team Opp FTM Avg', 'Home Team FT% Avg', 'Visitor Team FT% Avg', 'Home Team Opp FT% Avg', 'Visitor Team Opp FT% Avg', 'Home Team Fouls Avg', 'Visitor Team Fouls Avg', 'Home Team Opp Fouls Avg', 'Visitor Team Opp Fouls Avg', 'Home Team TO Avg', 'Visitor Team TO Avg', 'Home Team Opp TO Avg', 'Visitor Team Opp TO Avg']
for col in cols:
    modern[col] = modern[col].fillna(modern[col].mean())

In [ ]:
cols = ['Game ID', 'Date Number', 'Date', 'Game Type', 'Home Team', 'Visitor Team', 'Winner', 'Loser', 'Home Win', 'Home Team Points', 'Home Team Assists', 'Home Team Tot Rebounds', 'Home Team Off Rebounds', 'Home Team Def Rebounds', 'Home Team Blocks', 'Home Team Steals', 'Home Team FGA', 'Home Team FGM', 'Home Team FG%', 'Home Team 3PA', 'Home Team 3PM', 'Home Team 3P%', 'Home Team FTA', 'Home Team FTM', 'Home Team FT%', 'Home Team Fouls', 'Home Team TO', 'Visitor Team Points', 'Visitor Team Assists', 'Visitor Team Tot Rebounds', 'Visitor Team Off Rebounds', 'Visitor Team Def Rebounds', 'Visitor Team Blocks', 'Visitor Team Steals', 'Visitor Team FGA', 'Visitor Team FGM', 'Visitor Team FG%', 'Visitor Team 3PA', 'Visitor Team 3PM', 'Visitor Team 3P%', 'Visitor Team FTA', 'Visitor Team FTM', 'Visitor Team FT%', 'Visitor Team Fouls', 'Visitor Team TO', 'Playoff Game', 'Season', 'Home Team Points Avg', 'Visitor Team Points Avg', 'Home Team Opp Points Avg', 'Visitor Team Opp Points Avg', 'Home Team Assists Avg', 'Visitor Team Assists Avg', 'Home Team Opp Assists Avg', 'Visitor Team Opp Assists Avg', 'Home Team Tot Rebounds Avg', 'Visitor Team Tot Rebounds Avg', 'Home Team Opp Tot Rebounds Avg', 'Visitor Team Opp Tot Rebounds Avg', 'Home Team Off Rebounds Avg', 'Visitor Team Off Rebounds Avg', 'Home Team Opp Off Rebounds Avg', 'Visitor Team Opp Off Rebounds Avg', 'Home Team Def Rebounds Avg', 'Visitor Team Def Rebounds Avg', 'Home Team Opp Def Rebounds Avg', 'Visitor Team Opp Def Rebounds Avg', 'Home Team Blocks Avg', 'Visitor Team Blocks Avg', 'Home Team Opp Blocks Avg', 'Visitor Team Opp Blocks Avg', 'Home Team Steals Avg', 'Visitor Team Steals Avg', 'Home Team Opp Steals Avg', 'Visitor Team Opp Steals Avg', 'Home Team FGA Avg', 'Visitor Team FGA Avg', 'Home Team Opp FGA Avg', 'Visitor Team Opp FGA Avg', 'Home Team FGM Avg', 'Visitor Team FGM Avg', 'Home Team Opp FGM Avg', 'Visitor Team Opp FGM Avg', 'Home Team FG% Avg', 'Visitor Team FG% Avg', 'Home Team Opp FG% Avg', 'Visitor Team Opp FG% Avg', 'Home Team 3PA Avg', 'Visitor Team 3PA Avg', 'Home Team Opp 3PA Avg', 'Visitor Team Opp 3PA Avg', 'Home Team 3PM Avg', 'Visitor Team 3PM Avg', 'Home Team Opp 3PM Avg', 'Visitor Team Opp 3PM Avg', 'Home Team 3P% Avg', 'Visitor Team 3P% Avg', 'Home Team Opp 3P% Avg', 'Visitor Team Opp 3P% Avg', 'Home Team FTA Avg', 'Visitor Team FTA Avg', 'Home Team Opp FTA Avg', 'Visitor Team Opp FTA Avg', 'Home Team FTM Avg', 'Visitor Team FTM Avg', 'Home Team Opp FTM Avg', 'Visitor Team Opp FTM Avg', 'Home Team FT% Avg', 'Visitor Team FT% Avg', 'Home Team Opp FT% Avg', 'Visitor Team Opp FT% Avg', 'Home Team Fouls Avg', 'Visitor Team Fouls Avg', 'Home Team Opp Fouls Avg', 'Visitor Team Opp Fouls Avg', 'Home Team TO Avg', 'Visitor Team TO Avg', 'Home Team Opp TO Avg', 'Visitor Team Opp TO Avg']
modern = modern[cols]

In [ ]:
modern.to_csv("csvs/modern.csv")

In [ ]:
cols = ['Home Team FG%', 'Home Team 3P%', 'Home Team FT%', 'Visitor Team FG%', 'Visitor Team 3P%', 'Visitor Team FT%']
for col in cols:
    modern[col] = modern[col].fillna(0)

In [ ]:
columns = ['Points', 'Assists', 'Tot Rebounds', 
        'Off Rebounds', 'Def Rebounds', 'Blocks', 
        'Steals', 'FGA', 'FGM',
        '3PA', '3PM', 'FTA', 
        'FTM', 'Fouls', 'TO']
for col in columns:
    modern = calculate_rolling_stat_per_year(modern, col)
    
modern["Home Team FG% Avg"] = modern["Home Team FGM Avg"] / modern["Home Team FGA Avg"]
modern["Home Team FT% Avg"] = modern["Home Team FTM Avg"] / modern["Home Team FTA Avg"]
modern["Home Team 3P% Avg"] = modern["Home Team 3PM Avg"] / modern["Home Team 3PA Avg"]

modern["Visitor Team FG% Avg"] = modern["Visitor Team FGM Avg"] / modern["Visitor Team FGA Avg"]
modern["Visitor Team FT% Avg"] = modern["Visitor Team FTM Avg"] / modern["Visitor Team FTA Avg"]
modern["Visitor Team 3P% Avg"] = modern["Visitor Team 3PM Avg"] / modern["Visitor Team 3PA Avg"]

modern["Home Team Opp FG% Avg"] = modern["Home Team Opp FGM Avg"] / modern["Home Team Opp FGA Avg"]
modern["Home Team Opp FT% Avg"] = modern["Home Team Opp FTM Avg"] / modern["Home Team Opp FTA Avg"]
modern["Home Team Opp 3P% Avg"] = modern["Home Team Opp 3PM Avg"] / modern["Home Team Opp 3PA Avg"]

modern["Visitor Team Opp FG% Avg"] = modern["Visitor Team Opp FGM Avg"] / modern["Visitor Team Opp FGA Avg"]
modern["Visitor Team Opp FT% Avg"] = modern["Visitor Team Opp FTM Avg"] / modern["Visitor Team Opp FTA Avg"]
modern["Visitor Team Opp 3P% Avg"] = modern["Visitor Team Opp 3PM Avg"] / modern["Visitor Team Opp 3PA Avg"]

In [ ]:
def calculate_rolling_stat_per_year(df, stat_col, date_col="Date Number"):
    home_averages = []
    visitor_averages = []
    home_opp_averages = []
    visitor_opp_averages = []

    # Process year by year
    for year, group in df.groupby("Season"):
        stat_dict = {}  # Reset each year
        opp_stat_dict = {}
        for idx, row in group.iterrows():
            home_team = row["Home Team"]
            visitor_team = row["Visitor Team"]
            home_stat = row.get(f"Home Team {stat_col}", None)
            visitor_stat = row.get(f"Visitor Team {stat_col}", None)

            # Get current averages
            home_data = stat_dict.get(home_team, {"total": 0, "games": 0})
            visitor_data = stat_dict.get(visitor_team, {"total": 0, "games": 0})
            home_opp_data = opp_stat_dict.get(home_team, {"total": 0, "games": 0})
            visitor_opp_data = opp_stat_dict.get(visitor_team, {"total": 0, "games": 0})

            home_avg = (
                home_data["total"] / home_data["games"]
                if home_data["games"] > 0 else None
            )
            visitor_avg = (
                visitor_data["total"] / visitor_data["games"]
                if visitor_data["games"] > 0 else None
            )

            home_opp_avg = (
                home_opp_data["total"] / home_opp_data["games"]
                if home_opp_data["games"] > 0 else None
            )
            visitor_opp_avg = (
                visitor_opp_data["total"] / visitor_opp_data["games"]
                if visitor_opp_data["games"] > 0 else None
            )

            home_averages.append(home_avg)
            visitor_averages.append(visitor_avg)
            home_opp_averages.append(home_opp_avg)
            visitor_opp_averages.append(visitor_opp_avg)

            # Update totals *after* recording current averages
            stat_dict.setdefault(home_team, {"total": 0, "games": 0})
            stat_dict.setdefault(visitor_team, {"total": 0, "games": 0})
            opp_stat_dict.setdefault(home_team, {"total": 0, "games": 0})
            opp_stat_dict.setdefault(visitor_team, {"total": 0, "games": 0})

            if pd.notna(visitor_stat):
                stat_dict[visitor_team]["total"] += visitor_stat
                stat_dict[visitor_team]["games"] += 1
                opp_stat_dict[home_team]["total"] += visitor_stat
                opp_stat_dict[home_team]["games"] += 1

            if pd.notna(home_stat):
                stat_dict[home_team]["total"] += home_stat
                stat_dict[home_team]["games"] += 1
                opp_stat_dict[visitor_team]["total"] += home_stat
                opp_stat_dict[visitor_team]["games"] += 1
    # Add columns back to the original df (in order)
    df[f"Home Team {stat_col} Avg"] = home_averages
    df[f"Visitor Team {stat_col} Avg"] = visitor_averages
    df[f"Home Team Opp {stat_col} Avg"] = home_opp_averages
    df[f"Visitor Team Opp {stat_col} Avg"] = visitor_opp_averages

    return df

In [ ]:
def elo_change(home_elo, visitor_elo, margin, k=20):
    expected_home = 1 / (1 + 10 ** ((visitor_elo - home_elo) / 400))
    score = 1 if margin > 0 else 0
    elo_diff = home_elo - visitor_elo
    if margin < 0:
        margin = -margin
    mov_multiplier = ((margin + 3) ** 0.8) / (7.5 + 0.006 * elo_diff)
    # mov_multiplier = min(mov_multiplier, 2)
    return k * mov_multiplier * (score - expected_home)

In [ ]:
modern = pd.read_csv("modern_avgs_filled.csv")

In [ ]:
modern = get_all_elos(modern, {})

In [ ]:
modern = calculate_wl(modern)

In [ ]:
modern.to_csv("with_elo.csv")

In [ ]:
def get_all_elos(df: pd.DataFrame, team_elos: dict[tuple[str, int], float], base_elo: int =1500):
    base_elo = 1500

    for idx, game in df.iterrows():
        home_team = game["Home Team"]
        visitor_team = game["Visitor Team"]
        year = game["Season"]
        date_number = game["Date Number"]
        margin = game["Home Team Points"] - game["Visitor Team Points"]

        # Get raw (unadjusted) Elo from previous year or base
        raw_home_elo = team_elos.get((home_team, year - 1), base_elo)
        raw_visitor_elo = team_elos.get((visitor_team, year - 1), base_elo)

        # Check if first game of this team this year
        is_first_home_game = (home_team, year) not in team_elos
        is_first_visitor_game = (visitor_team, year) not in team_elos

        # Apply seasonal decay BEFORE storing
        if is_first_home_game:
            home_elo = 0.75 * raw_home_elo + 0.25 * base_elo
            team_elos[(home_team, year)] = home_elo  # initialize season
        else:
            home_elo = team_elos[(home_team, year)]

        if is_first_visitor_game:
            visitor_elo = 0.75 * raw_visitor_elo + 0.25 * base_elo
            team_elos[(visitor_team, year)] = visitor_elo
        else:
            visitor_elo = team_elos[(visitor_team, year)]

        df.loc[idx, "Home Team ELO"] = home_elo
        df.loc[idx, "Visitor Team ELO"] = visitor_elo
        
        # Calculate Elo change
        delta = elo_change(home_elo+100, visitor_elo, margin)
        # Update Elo for next game
        team_elos[(home_team, year)] += delta
        team_elos[(visitor_team, year)] -= delta
    # print(team_elos)
    return df

In [ ]:
def calculate_wl(df, date_col="Date Number"):
    home_wl = []
    visitor_wl = []
    home_tot = []
    home_w = []
    home_l = []
    visitor_tot = []
    visitor_w = []
    visitor_l = []
    # Process year by year
    for year, group in df.groupby("Season"):
        stat_dict = {}  # Reset each year
        for idx, row in group.iterrows():
            home_team = row["Home Team"]
            visitor_team = row["Visitor Team"]
            home_win = row.get("Home Win", None)

            # Get current averages
            home_data = stat_dict.get(home_team, {"wins": 0, "games": 0})
            visitor_data = stat_dict.get(visitor_team, {"wins": 0, "games": 0})

            home_avg = (
                home_data["wins"] / home_data["games"]
                if home_data["games"] > 0 else 0.5
            )
            visitor_avg = (
                visitor_data["wins"] / visitor_data["games"]
                if visitor_data["games"] > 0 else 0.5
            )


            home_wl.append(home_avg)
            visitor_wl.append(visitor_avg)
            home_tot.append(home_data["games"])
            home_w.append(home_data["wins"])
            home_l.append(home_data["games"] - home_data["wins"])
            visitor_tot.append(visitor_data["games"])
            visitor_w.append(visitor_data["wins"])
            visitor_l.append(visitor_data["games"] - visitor_data["wins"])

            # Update totals *after* recording current averages
            stat_dict.setdefault(home_team, {"wins": 0, "games": 0})
            stat_dict.setdefault(visitor_team, {"wins": 0, "games": 0})

            if pd.notna(home_win):
                stat_dict[home_team]["wins"] += home_win
                stat_dict[home_team]["games"] += 1
                stat_dict[visitor_team]["wins"] += (1 - home_win)
                stat_dict[visitor_team]["games"] += 1

    # Add columns back to the original df (in order)
    df["Home Team Total Games"] = home_tot
    df["Home Team W"] = home_w
    df["Home Team L"] = home_l
    df["Home Team W/L%"] = home_wl
    df["Visitor Team Total Games"] = visitor_tot
    df["Visitor Team W"] = visitor_w
    df["Visitor Team L"] = visitor_l
    df["Visitor Team W/L%"] = visitor_wl

    return df

In [ ]:
all_games = pd.read_csv("csvs/all_games.csv")

In [ ]:
all_games.to_csv("csvs/all_games.csv")

In [ ]:
modern = calculate_wl(modern)
modern.to_csv("csvs/results_based_avgs.csv")

In [ ]:
all_games.loc[all_games["Game ID"] == 69929, "Loser"] = "New Orleans Pelicans"
all_games.loc[all_games["Game ID"] == 69929, "Home Win"] = 0

In [ ]:
all_games = all_games[all_games["Game ID"] != 69789]

In [ ]:
miss = pd.read_csv("csvs/with_missing_games.csv")

In [ ]:
all_games = all_games[['Game ID', 'Date Number', 'Date', 'Game Type', 'Home Team', 'Visitor Team', 'Winner', 'Loser', 'Home Win', 'Home Team Total Games', 'Visitor Team Total Games', 'Home Team W', 'Home Team L', 'Visitor Team W', 'Visitor Team L', 'Home Team Points', 'Home Team Assists', 'Home Team Tot Rebounds', 'Home Team Off Rebounds', 'Home Team Def Rebounds', 'Home Team Blocks', 'Home Team Steals', 'Home Team FGA', 'Home Team FGM', 'Home Team FG%', 'Home Team 3PA', 'Home Team 3PM', 'Home Team 3P%', 'Home Team FTA', 'Home Team FTM', 'Home Team FT%', 'Home Team Fouls', 'Home Team TO', 'Visitor Team Points', 'Visitor Team Assists', 'Visitor Team Tot Rebounds', 'Visitor Team Off Rebounds', 'Visitor Team Def Rebounds', 'Visitor Team Blocks', 'Visitor Team Steals', 'Visitor Team FGA', 'Visitor Team FGM', 'Visitor Team FG%', 'Visitor Team 3PA', 'Visitor Team 3PM', 'Visitor Team 3P%', 'Visitor Team FTA', 'Visitor Team FTM', 'Visitor Team FT%', 'Visitor Team Fouls', 'Visitor Team TO', 'Playoff Game', 'Season']]

In [ ]:
hornets = all_games[(all_games["Season"] == 2025) & (all_games["Playoff Game"] == 0)]
hornets = hornets[(hornets["Winner"] == "New Orleans Pelicans") | (hornets["Loser"] == "New Orleans Pelicans")]
print(len(hornets))
hornets.to_csv("csvs/test_hornets.csv")

In [ ]:
modern = modern.fillna(modern.mean(numeric_only=True))

In [ ]:
modern.isna().sum().sum()

In [69]:
modern = modern.loc[:, ~modern.columns.str.contains("^Unnamed")]

In [ ]:
modern.to_csv("with_wl.csv")

In [ ]:
modern = pd.read_csv("csvs/with_wl.csv")

In [ ]:
margins = modern[['Game ID', 'Date Number', 'Date', 'Game Type', 'Home Team', 'Visitor Team', 'Winner', 'Loser', 'Home Win', 'Season']].copy()
cols = ['Points', 'Assists', 'Tot Rebounds', 
        'Off Rebounds', 'Def Rebounds', 'Blocks', 
        'Steals', 'FGA', 'FGM', 'FG%',
        '3PA', '3PM', '3P%', 'FTA', 
        'FTM', 'FT%', 'Fouls', 'TO']
for col in cols:
    margins[f"Home Team {col} Margin Avg"] = modern[f"Home Team {col} Avg"] - modern[f"Home Team Opp {col} Avg"]
    margins[f"Visitor Team {col} Margin Avg"] = modern[f"Visitor Team {col} Avg"] - modern[f"Visitor Team Opp {col} Avg"]
margins["ELO Margin"] = modern["Home Team ELO"] - modern["Visitor Team ELO"]

In [ ]:
margins = modern[['Game ID', 'Date Number', 'Date', 'Game Type', 'Home Team', 'Visitor Team', 'Winner', 'Loser', 'Home Win', 'Season']].copy()
cols = ['Points', 'Assists', 'Tot Rebounds', 
        'Off Rebounds', 'Def Rebounds', 'Blocks', 
        'Steals', 'FGA', 'FGM', 'FG%',
        '3PA', '3PM', '3P%', 'FTA', 
        'FTM', 'FT%', 'Fouls', 'TO']
for col in cols:
    margins[f"{col} Avg Margin"] = modern[f"Home Team {col} Avg"] - modern[f"Visitor Team {col} Avg"]
    margins[f"Opp {col} Avg Margin"] = modern[f"Home Team Opp {col} Avg"] - modern[f"Visitor Team Opp {col} Avg"]
margins["ELO Margin"] = modern["Home Team ELO"] - modern["Visitor Team ELO"]

In [ ]:
margins.to_csv("csvs/avg_margins_both_teams.csv")

In [ ]:
locs = locs.loc[:, ~locs.columns.str.contains("^Unnamed")]

In [ ]:
margins = pd.read_csv("csvs/avg_margins_ea_team.csv")

In [ ]:
cols = ['Points', 'Assists', 'Tot Rebounds', 
        'Off Rebounds', 'Def Rebounds', 'Blocks', 
        'Steals', 'FGA', 'FGM', 'FG%',
        '3PA', '3PM', '3P%', 'FTA', 
        'FTM', 'FT%', 'Fouls', 'TO']
for col in cols:
    margins[f"{col} Margin Avg Diff"] = margins[f"Home Team {col} Margin Avg"] - margins[f"Visitor Team {col} Margin Avg"]

In [ ]:
locs = modern[['Game ID', 'Date Number', 'Date', 'Game Type', 'Home Team', 'Visitor Team', 'Winner', 'Loser', 'Home Win', 'Season']].copy()
columns = ['Points', 'Assists', 'Tot Rebounds', 
        'Off Rebounds', 'Def Rebounds', 'Blocks', 
        'Steals', 'FGA', 'FGM', 'FG%',
        '3PA', '3PM', '3P%', 'FTA', 
        'FTM', 'FT%', 'Fouls', 'TO']
for col in columns:
    locs = calculate_location_avg(locs, modern, col)

In [ ]:
locs.to_csv("csvs/results_based_avgs.csv")

In [ ]:
def calculate_location_avg(locs, df, stat_col, date_col="Date Number"):
    home_averages = []
    visitor_averages = []
    home_opp_averages = []
    visitor_opp_averages = []

    # Process year by year
    for year, group in df.groupby("Season"):
        home_stat_dict = {}
        home_opp_stat_dict = {}
        visitor_stat_dict = {}
        visitor_opp_stat_dict = {}
        for idx, row in group.iterrows():
            home_team = row["Home Team"]
            visitor_team = row["Visitor Team"]
            home_stat = row.get(f"Home Team {stat_col}", None)
            visitor_stat = row.get(f"Visitor Team {stat_col}", None)

            # Get current averages
            home_data = home_stat_dict.get(home_team, {"total": 0, "games": 0})
            visitor_data = visitor_stat_dict.get(visitor_team, {"total": 0, "games": 0})
            home_opp_data = home_opp_stat_dict.get(home_team, {"total": 0, "games": 0})
            visitor_opp_data = visitor_opp_stat_dict.get(visitor_team, {"total": 0, "games": 0})

            home_avg = (
                home_data["total"] / home_data["games"]
                if home_data["games"] > 0 else None
            )
            visitor_avg = (
                visitor_data["total"] / visitor_data["games"]
                if visitor_data["games"] > 0 else None
            )

            home_opp_avg = (
                home_opp_data["total"] / home_opp_data["games"]
                if home_opp_data["games"] > 0 else None
            )
            visitor_opp_avg = (
                visitor_opp_data["total"] / visitor_opp_data["games"]
                if visitor_opp_data["games"] > 0 else None
            )

            home_averages.append(home_avg)
            visitor_averages.append(visitor_avg)
            home_opp_averages.append(home_opp_avg)
            visitor_opp_averages.append(visitor_opp_avg)

            # Update totals *after* recording current averages
            home_stat_dict.setdefault(home_team, {"total": 0, "games": 0})
            visitor_stat_dict.setdefault(visitor_team, {"total": 0, "games": 0})
            home_opp_stat_dict.setdefault(home_team, {"total": 0, "games": 0})
            visitor_opp_stat_dict.setdefault(visitor_team, {"total": 0, "games": 0})

            if pd.notna(visitor_stat):
                visitor_stat_dict[visitor_team]["total"] += visitor_stat
                visitor_stat_dict[visitor_team]["games"] += 1
                home_opp_stat_dict[home_team]["total"] += visitor_stat
                home_opp_stat_dict[home_team]["games"] += 1

            if pd.notna(home_stat):
                home_stat_dict[home_team]["total"] += home_stat
                home_stat_dict[home_team]["games"] += 1
                visitor_opp_stat_dict[visitor_team]["total"] += home_stat
                visitor_opp_stat_dict[visitor_team]["games"] += 1

    # Add columns back to the original df (in order)
    locs[f"Home Team Home {stat_col} Avg"] = home_averages
    locs[f"Visitor Team Visitor {stat_col} Avg"] = visitor_averages
    locs[f"Home Team Home Opp {stat_col} Avg"] = home_opp_averages
    locs[f"Visitor Team Visitor Opp {stat_col} Avg"] = visitor_opp_averages

    return locs

In [ ]:
locs = calculate_loc_wl(locs)

In [ ]:
def calculate_loc_wl(df, date_col="Date Number"):
    home_wl = []
    visitor_wl = []
    home_tot = []
    home_w = []
    home_l = []
    visitor_tot = []
    visitor_w = []
    visitor_l = []
    # Process year by year
    for year, group in df.groupby("Season"):
        home_stat_dict = {}
        visitor_stat_dict = {}
        for idx, row in group.iterrows():
            home_team = row["Home Team"]
            visitor_team = row["Visitor Team"]
            home_win = row.get("Home Win", None)

            # Get current averages
            home_data = home_stat_dict.get(home_team, {"wins": 0, "games": 0})
            visitor_data = visitor_stat_dict.get(visitor_team, {"wins": 0, "games": 0})

            home_avg = (
                home_data["wins"] / home_data["games"]
                if home_data["games"] > 0 else 0.6
            )
            visitor_avg = (
                visitor_data["wins"] / visitor_data["games"]
                if visitor_data["games"] > 0 else 0.4
            )


            home_wl.append(home_avg)
            visitor_wl.append(visitor_avg)
            home_tot.append(home_data["games"])
            home_w.append(home_data["wins"])
            home_l.append(home_data["games"] - home_data["wins"])
            visitor_tot.append(visitor_data["games"])
            visitor_w.append(visitor_data["wins"])
            visitor_l.append(visitor_data["games"] - visitor_data["wins"])

            # Update totals *after* recording current averages
            home_stat_dict.setdefault(home_team, {"wins": 0, "games": 0})
            visitor_stat_dict.setdefault(visitor_team, {"wins": 0, "games": 0})

            if pd.notna(home_win):
                home_stat_dict[home_team]["wins"] += home_win
                home_stat_dict[home_team]["games"] += 1
                visitor_stat_dict[visitor_team]["wins"] += (1 - home_win)
                visitor_stat_dict[visitor_team]["games"] += 1

    # Add columns back to the original df (in order)
    df["Home Team Home Games"] = home_tot
    df["Home Team Home W"] = home_w
    df["Home Team Home L"] = home_l
    df["Home Team Home W/L%"] = home_wl
    df["Visitor Team Visitor Games"] = visitor_tot
    df["Visitor Team Visitor W"] = visitor_w
    df["Visitor Team Visitor L"] = visitor_l
    df["Visitor Team Visitor W/L%"] = visitor_wl

    return df

In [ ]:
locs = locs.fillna(locs.mean(numeric_only=True))

In [ ]:
wls = modern[['Game ID', 'Date Number', 'Date', 'Game Type', 'Home Team', 'Visitor Team', 'Winner', 'Loser', 'Home Win', 'Season']].copy()
columns = ['Points', 'Assists', 'Tot Rebounds', 
        'Off Rebounds', 'Def Rebounds', 'Blocks', 
        'Steals', 'FGA', 'FGM', 'FG%',
        '3PA', '3PM', '3P%', 'FTA', 
        'FTM', 'FT%', 'Fouls', 'TO']
for col in columns:
    wls = calculate_wl_avg(locs, modern, col)

In [ ]:
players = pd.read_csv('archive/PlayerStatistics.csv')

In [ ]:
# Create rename mapping
rename_map = {
    "win": "Win",
    "personId": "Player ID",
    "gameId": "Game ID",
    "gameDate": "Date",
    "gameType": "Game Type",
    "home": "Home",
    "numMinutes": "Minutes",
    "points": "Points",
    "assists": "Assists",
    "reboundsTotal": "Tot Rebounds",
    "reboundsOffensive": "Off Rebounds",
    "reboundsDefensive": "Def Rebounds",
    "blocks": "Blocks",
    "steals": "Steals",
    "fieldGoalsAttempted": "FGA",
    "fieldGoalsMade": "FGM",
    "fieldGoalsPercentage": "FG%",
    "threePointersAttempted": "3PA",
    "threePointersMade": "3PM",
    "threePointersPercentage": "3P%",
    "freeThrowsAttempted": "FTA",
    "freeThrowsMade": "FTM",
    "freeThrowsPercentage": "FT%",
    "foulsPersonal": "Fouls",
    "turnovers": "TO",
    "plusMinusPoints": "Plus Minus",
}

# Apply rename
players = players.rename(columns=rename_map)


In [ ]:
players["Player Name"] = players["firstName"] + " " + players["lastName"]
players["Team Name"] = players["playerteamCity"] + " " + players["playerteamName"]
players["Opp Team Name"] = players["opponentteamCity"] + " " + players["opponentteamName"]

In [ ]:
players = players[['Player ID', 'Game ID', 'Player Name', 'Team Name', 'Opp Team Name', 'Date', 'Date Number', 'Season','Game Type', 'Win', 'Home', 'MP', 'PTS', 'AST',
       'BLK', 'STL', 'FGA', 'FG', 'FG%', '3PA', '3P', '3P%', 'FTA', 'FT',
       'FT%', 'DRB', 'ORB', 'TRB', 'PF', 'TOV', '+/-']]

In [ ]:
players = pd.read_csv("csvs/all_players.csv")

In [ ]:
# --- Players DF ---
players["MatchKey"] = (
    players[["Team Name", "Opp Team Name"]].apply(sorted, axis=1).str.join("_")
    + "_" + players["Date Number"].astype(str)
)

# --- Games DF ---
all_games["MatchKey"] = (
    all_games[["Home Team", "Visitor Team"]].apply(sorted, axis=1).str.join("_")
    + "_" + all_games["Date Number"].astype(str)
)

players = players.drop(columns=["Game ID"])
players = players.drop(columns=["Date Number"])
players = players.drop(columns=["Date"])
players = players.merge(
    all_games[["Game ID", "Date Number", "Date", "MatchKey", "Season"]],
    on="MatchKey",
    how="left"
)

# Drop the helper column
players = players.drop(columns=["MatchKey"])


In [ ]:
players["Game ID"] = players["Game ID"].astype(int)

In [ ]:
players = players[(players["Game Type"] == "Regular Season") | (players["Game Type"] == "Playoffs")]

In [ ]:
players.to_csv("csvs/all_players.csv")

In [ ]:
played = players.fillna(0)
played = played[played["MP"] > 0]
played.to_csv("csvs/players_game_time.csv")

In [ ]:
players = players.sort_values(by="Date Number").reset_index(drop=True)

In [ ]:
# Create rename mapping
rename_map = {
    "Minutes": "MP",
    "Points": "PTS",
    "Assists": "AST",
    "Tot Rebounds": "TRB",
    "Off Rebounds": "ORB",
    "Def Rebounds": "DRB",
    "Blocks": "BLK",
    "Steals": "STL",
    "fieldGoalsAttempted": "FGA",
    "FGM": "FG",
    "fieldGoalsPercentage": "FG%",
    "threePointersAttempted": "3PA",
    "3PM": "3P",
    "threePointersPercentage": "3P%",
    "freeThrowsAttempted": "FTA",
    "FTM": "FT",
    "freeThrowsPercentage": "FT%",
    "Fouls": "PF",
    "Plus Minus": "+/-",
    "TO": "TOV"
}


# Apply rename
players = players.rename(columns=rename_map)


In [ ]:
players.columns

In [ ]:
players['Player Name'] = players['Player Name'].replace("AJ Green", "A.J. Green")


In [ ]:
bs = pd.read_csv("MissingGames/jazz(wizards).csv")
bs = bs.fillna(0)
new_df_list = []
for idx, row in bs.iterrows():
    player = row["Starters"]
    player = player.replace("ć", "c")
    player = player.replace("č", "c")
    player = player.replace("ü", "u")
    player = player.replace("é", "e")
    player = player.replace("Ş", "S")
    player = player.replace("ñ", "n")
    print(player)
    if player == "Team Totals":
        continue
    mp = row["MP"]
    if mp != 'Did Not Play' and mp != "Did Not Dress":
        matches = players[(players["Player Name"] == player) & (players["Game ID"] > 69232)]
        pid = matches.iloc[0]["Player ID"]
        split = mp.split(":")
        mp = float(split[0] + "." + split[1])
        new_row = {
            'Player ID': pid,
            'Game ID': 0,  # placeholder, update with authoritative Game ID later
            'Player Name': player,
            'Opp Team Name': "Washington Wizards",
            'Team Name': "Utah Jazz",
            'Date': '',                # assuming you have this in `row`
            'Date Number': 739329,
            'Game Type': 'Regular Season',      # or hardcode if needed
            'Win': 1,
            'Home': 1,
            'MP': mp,
            'PTS': float(row['PTS']),
            'AST': float(row['AST']),
            'BLK': float(row['BLK']),
            'STL': float(row['STL']),
            'FGA': float(row['FGA']),
            'FG': float(row['FG']),       # field goals made
            'FG%': float(row['FG%']),
            '3PA': float(row['3PA']),
            '3P': float(row['3P']),       # 3-pointers made
            '3P%': float(row['3P%']),
            'FTA': float(row['FTA']),
            'FT': float(row['FT']),       # free throws made
            'FT%': float(row['FT%']),
            'DRB': float(row['DRB']),
            'ORB': float(row['ORB']),
            'TRB': float(row['TRB']),
            'PF': float(row['PF']),       # personal fouls
            'TOV': float(row['TOV']),
            '+/-': float(row['+/-']),
        }
        new_df_list.append(new_row)
    
    # Create the new DataFrame
new_df = pd.DataFrame(new_df_list)
players = pd.concat([players, new_df], ignore_index=True)


In [ ]:
def calculate_result_based_avg(results, df, stat_col, date_col="Date Number"):
    home_w_averages = []
    home_l_averages = []
    home_w_opp_averages = []
    home_l_opp_averages = []
    
    visitor_w_averages = []
    visitor_l_averages = []
    visitor_w_opp_averages = []
    visitor_l_opp_averages = []

    # Process year by year
    for year, group in df.groupby("Season"):
        home_w_stat_dict = {}
        home_l_stat_dict = {}
        home_w_opp_stat_dict = {}
        home_l_opp_stat_dict = {}
        
        visitor_w_stat_dict = {}
        visitor_l_stat_dict = {}
        visitor_w_opp_stat_dict = {}
        visitor_l_opp_stat_dict = {}
        for idx, row in group.iterrows():
            home_team = row["Home Team"]
            visitor_team = row["Visitor Team"]
            home_win = row["Home Win"]
            home_stat = row.get(f"Home Team {stat_col}", None)
            visitor_stat = row.get(f"Visitor Team {stat_col}", None)

            # Get current averages
            home_w_data = home_w_stat_dict.get(home_team, {"total": 0, "games": 0})
            home_l_data = home_l_stat_dict.get(home_team, {"total": 0, "games": 0})
            home_w_opp_data = home_w_opp_stat_dict.get(home_team, {"total": 0, "games": 0})
            home_l_opp_data = home_l_opp_stat_dict.get(home_team, {"total": 0, "games": 0})
            
            visitor_w_data = visitor_w_stat_dict.get(visitor_team, {"total": 0, "games": 0})
            visitor_l_data = visitor_l_stat_dict.get(visitor_team, {"total": 0, "games": 0})
            visitor_w_opp_data = visitor_w_opp_stat_dict.get(visitor_team, {"total": 0, "games": 0})
            visitor_l_opp_data = visitor_l_opp_stat_dict.get(visitor_team, {"total": 0, "games": 0})

            home_w_avg = (
                home_w_data["total"] / home_w_data["games"]
                if home_w_data["games"] > 0 else None
            )

            home_l_avg = (
                home_l_data["total"] / home_l_data["games"]
                if home_l_data["games"] > 0 else None
            )

            home_w_opp_avg = (
                home_w_opp_data["total"] / home_w_opp_data["games"]
                if home_w_opp_data["games"] > 0 else None
            )

            home_l_opp_avg = (
                home_l_opp_data["total"] / home_l_opp_data["games"]
                if home_l_opp_data["games"] > 0 else None
            )
            
            visitor_w_avg = (
                visitor_w_data["total"] / visitor_w_data["games"]
                if visitor_w_data["games"] > 0 else None
            )

            visitor_l_avg = (
                visitor_l_data["total"] / visitor_l_data["games"]
                if visitor_l_data["games"] > 0 else None
            )

            visitor_w_opp_avg = (
                visitor_w_opp_data["total"] / visitor_w_opp_data["games"]
                if visitor_w_opp_data["games"] > 0 else None
            )

            visitor_l_opp_avg = (
                visitor_l_opp_data["total"] / visitor_l_opp_data["games"]
                if visitor_l_opp_data["games"] > 0 else None
            )
            
            home_w_averages.append(home_w_avg)
            visitor_w_averages.append(visitor_w_avg)
            home_w_opp_averages.append(home_w_opp_avg)
            visitor_w_opp_averages.append(visitor_w_opp_avg)

            home_l_averages.append(home_l_avg)
            visitor_l_averages.append(visitor_l_avg)
            home_l_opp_averages.append(home_l_opp_avg)
            visitor_l_opp_averages.append(visitor_l_opp_avg)
            
            # Update totals *after* recording current averages
            home_w_stat_dict.setdefault(home_team, {"total": 0, "games": 0})
            visitor_w_stat_dict.setdefault(visitor_team, {"total": 0, "games": 0})
            home_w_opp_stat_dict.setdefault(home_team, {"total": 0, "games": 0})
            visitor_w_opp_stat_dict.setdefault(visitor_team, {"total": 0, "games": 0})

            home_l_stat_dict.setdefault(home_team, {"total": 0, "games": 0})
            visitor_l_stat_dict.setdefault(visitor_team, {"total": 0, "games": 0})
            home_l_opp_stat_dict.setdefault(home_team, {"total": 0, "games": 0})
            visitor_l_opp_stat_dict.setdefault(visitor_team, {"total": 0, "games": 0})

            if home_win == 1:
                visitor_l_stat_dict[visitor_team]["total"] += visitor_stat
                visitor_l_stat_dict[visitor_team]["games"] += 1
                home_w_opp_stat_dict[home_team]["total"] += visitor_stat
                home_w_opp_stat_dict[home_team]["games"] += 1
                visitor_l_opp_stat_dict[visitor_team]["total"] += home_stat
                visitor_l_opp_stat_dict[visitor_team]["games"] += 1
                home_w_stat_dict[home_team]["total"] += home_stat
                home_w_stat_dict[home_team]["games"] += 1
            else:
                visitor_w_stat_dict[visitor_team]["total"] += visitor_stat
                visitor_w_stat_dict[visitor_team]["games"] += 1
                home_l_opp_stat_dict[home_team]["total"] += visitor_stat
                home_l_opp_stat_dict[home_team]["games"] += 1
                visitor_w_opp_stat_dict[visitor_team]["total"] += home_stat
                visitor_w_opp_stat_dict[visitor_team]["games"] += 1
                home_l_stat_dict[home_team]["total"] += home_stat
                home_l_stat_dict[home_team]["games"] += 1

    # Add columns back to the original df (in order)
    results[f"Home Team {stat_col} Avg in Wins"] = home_w_averages
    results[f"Visitor Team {stat_col} Avg in Wins"] = visitor_w_averages
    results[f"Home Team Opp {stat_col} Avg in Wins"] = home_w_opp_averages
    results[f"Visitor Team Opp {stat_col} Avg in Wins"] = visitor_w_opp_averages

    results[f"Home Team {stat_col} Avg in Losses"] = home_l_averages
    results[f"Visitor Team {stat_col} Avg in Losses"] = visitor_l_averages
    results[f"Home Team Opp {stat_col} Avg in Losses"] = home_l_opp_averages
    results[f"Visitor Team Opp {stat_col} Avg in Losses"] = visitor_l_opp_averages

    return results

In [ ]:
results = modern[['Game ID', 'Date Number', 'Date', 'Game Type', 'Home Team', 'Visitor Team', 'Winner', 'Loser', 'Home Win', 'Season']].copy()
columns = ['Points', 'Assists', 'Tot Rebounds', 
        'Off Rebounds', 'Def Rebounds', 'Blocks', 
        'Steals', 'FGA', 'FGM', 'FG%',
        '3PA', '3PM', '3P%', 'FTA', 
        'FTM', 'FT%', 'Fouls', 'TO']
for col in columns:
    results = calculate_result_based_avg(results, modern, col)

In [ ]:
results.to_csv('csvs/results_based_avgs.csv')

In [ ]:
print(list(players.columns))

In [ ]:
print(list(modern.columns))

In [ ]:
results = results.fillna(results.mean(numeric_only=True))

In [ ]:
results_based_avgs = pd.read_csv("csvs/results_based_avgs.csv")

results_based_avgs.columns = results_based_avgs.columns.str.replace("Points", "PTS")
results_based_avgs.columns = results_based_avgs.columns.str.replace("Assists", "AST")
results_based_avgs.columns = results_based_avgs.columns.str.replace("Blocks", "BLK")
results_based_avgs.columns = results_based_avgs.columns.str.replace("Steals", "STL")
results_based_avgs.columns = results_based_avgs.columns.str.replace("Off Rebounds", "ORB")
results_based_avgs.columns = results_based_avgs.columns.str.replace("Def Rebounds", "DRB")
results_based_avgs.columns = results_based_avgs.columns.str.replace("Tot Rebounds", "TRB")
results_based_avgs.columns = results_based_avgs.columns.str.replace("Fouls", "PF")
results_based_avgs.columns = results_based_avgs.columns.str.replace("TOVV", "TOV")
results_based_avgs.columns = results_based_avgs.columns.str.replace("FGM", "FG")
results_based_avgs.columns = results_based_avgs.columns.str.replace("3PM", "3P")
results_based_avgs.columns = results_based_avgs.columns.str.replace("FTM", "FT")

results_based_avgs = results_based_avgs.loc[:, ~results_based_avgs.columns.str.contains("^Unnamed")]

results_based_avgs.to_csv("csvs/results_based_avgs.csv")

print(list(results_based_avgs.columns))

In [ ]:
(PTS + REB + AST + STL + BLK − Missed FG − Missed FT - TO) / GP

In [ ]:
players = pd.read_csv("csvs/all_players.csv")
players["EFF"] = players["PTS"] + players["TRB"] + players["AST"] + players["STL"] + players["BLK"] + players["FG"] - players["FGA"] + players["FT"] - players["FTA"] - players["TOV"]
players = players.loc[:, ~players.columns.str.contains("^Unnamed")]
players.to_csv("csvs/all_players.csv")

In [ ]:
def calculate_eff_avg(df, date_col="Date Number"):
    averages = []

    # Process year by year
    for year, group in df.groupby("Season"):
        stat_dict = {}
        for idx, row in group.iterrows():
            player = row["Player Name"]
            stat = row.get("EFF", None)

            # Get current averages
            data = stat_dict.get(player, {"total": 0, "games": 0})

            avg = (
                data["total"] / data["games"]
                if data["games"] > 0 else None
            )

            averages.append(avg)
            
            # Update totals *after* recording current averages
            stat_dict.setdefault(player, {"total": 0, "games": 0})

            if pd.notna(stat):
                stat_dict[player]["total"] += stat
                stat_dict[player]["games"] += 1

    df[f"EFF Avg"] = averages

    return df

In [ ]:
def calculate_career_eff_avg(df, date_col="Date Number"):
    averages = []
    stat_dict = {}
    for idx, row in df.iterrows():
        player = row["Player Name"]
        stat = row.get("EFF", None)

        # Get current averages
        data = stat_dict.get(player, {"total": 0, "games": 0})

        avg = (
            data["total"] / data["games"]
            if data["games"] > 0 else None
        )

        averages.append(avg)
        
        # Update totals *after* recording current averages
        stat_dict.setdefault(player, {"total": 0, "games": 0})

        if pd.notna(stat):
            stat_dict[player]["total"] += stat
            stat_dict[player]["games"] += 1

    df[f"Career EFF Avg"] = averages

    return df

In [ ]:
players = calculate_career_eff_avg(players)
players.to_csv("csvs/all_players.csv")

In [ ]:
players.to_csv("csvs/all_players.csv")

In [ ]:
players["EFF Avg"] = players["EFF Avg"].fillna(players["Career EFF Avg"])

In [ ]:
players = players.rename(columns={"EFF Avg": "Season EFF Avg"})

In [ ]:
players["Season EFF Avg"] = players["Season EFF Avg"].fillna(players["Season EFF Avg"].mean())
players["Career EFF Avg"] = players["Career EFF Avg"].fillna(players["Career EFF Avg"].mean())

In [ ]:
players.isna().sum()

In [688]:
def calculate_team_effs(df):
    season_effs = []
    career_effs = []
    for idx, row in df.iterrows():
        home = row["Home Team"]
        visitor = row["Visitor Team"]
        game_id = row["Game ID"]
        game_players = modern_players[modern_players["Game ID"] == game_id]
        home_players = game_players[game_players["Team Name"] == home]
        visitor_players = game_players[game_players["Team Name"] == visitor]
        home_min = home_players["MP Avg"].sum()
        visitor_min = visitor_players["MP Avg"].sum()
        df.loc[idx, "Home Team Tot New Season EFF Avg"] = home_players["Season EFF Avg"].sum() * 240 / home_min if home_min > 0 else 0
        df.loc[idx, "Home Team Tot New Career EFF Avg"] = home_players["Career EFF Avg"].sum() * 240 / home_min if home_min > 0 else 0
        df.loc[idx, "Visitor Team Tot New Season EFF Avg"] = visitor_players["Season EFF Avg"].sum() * 240 / visitor_min if visitor_min > 0 else 0
        df.loc[idx, "Visitor Team Tot New Career EFF Avg"] = visitor_players["Career EFF Avg"].sum() * 240 / visitor_min if visitor_min > 0 else 0

    return df

In [689]:
modern = calculate_team_effs(modern)
modern.to_csv("csvs/modern.csv")

In [690]:
modern.to_csv("csvs/modern.csv")

In [ ]:
print(list(modern.columns))

In [ ]:
margin1 = pd.read_csv("csvs/avg_margins_btw_teams.csv")
margin2 = pd.read_csv("csvs/avg_margins_each_team.csv")
margin1["W/L% Margin"] = modern["Home Team W/L%"] - modern["Visitor Team W/L%"]
margin1["Tot Season EFF Avg Margin"] = modern['Home Team Tot Season EFF Avg'] - modern['Visitor Team Tot Season EFF Avg']
margin1["Tot Career EFF Avg Margin"] = modern['Home Team Tot Career EFF Avg'] - modern['Visitor Team Tot Career EFF Avg']
margin2["W/L% Margin"] = modern["Home Team W/L%"] - modern["Visitor Team W/L%"]
margin2["Tot Season EFF Avg Margin"] = modern['Home Team Tot Season EFF Avg'] - modern['Visitor Team Tot Season EFF Avg']
margin2["Tot Career EFF Avg Margin"] = modern['Home Team Tot Career EFF Avg'] - modern['Visitor Team Tot Career EFF Avg']
margin1.to_csv("csvs/avg_margins_btw_teams.csv")
margin2.to_csv("csvs/avg_margins_each_team.csv")

In [ ]:
margin2 = margin2.loc[:, ~margin2.columns.str.contains("^Unnamed")]
margin2.to_csv("csvs/avg_margins_each_team.csv")

In [ ]:
df["PTS Margin"] = df["PTS Avg Margin"] - df["Opp PTS Avg Margin"]
df["PTS Margin"] = df["Home Team PTS Margin Avg"] - df["Visitor Team PTS Margin Avg"]

In [ ]:
model_categories = margin2[['Home Team', 'Visitor Team', 'Home Team PTS Margin Avg', 'Visitor Team PTS Margin Avg','W/L% Margin', 'ELO Margin',
              'Tot Season EFF Avg Margin', 'Tot Career EFF Avg Margin', 'Game ID', 'Date Number', 'Season']].copy()
model_categories['PTS Avg Margin'] = margin1['PTS Avg Margin'].copy()
model_categories["Opp PTS Avg Margin"] = margin1["Opp PTS Avg Margin"].copy()
model_categories[['Home Team PTS', 'Visitor Team PTS', 'Home Win', 'Home Team PTS Avg', 'Visitor Team PTS Avg']] = modern[['Home Team PTS', 'Visitor Team PTS', 'Home Win', 'Home Team PTS Avg', 'Visitor Team PTS Avg']].copy()
model_categories.to_csv("csvs/model_categories.csv")

In [ ]:
print(list(modern.columns))

In [ ]:
last_games = {}
for idx, game in modern.iterrows():
    home = game["Home Team"]
    visitor = game["Visitor Team"]
    date_num = game["Date Number"]
    home_last = last_games.get(home, 0)
    visitor_last = last_games.get(visitor, 0)
    modern.loc[idx, "Home Team Rest Days"] = min(3, date_num - home_last-1)
    modern.loc[idx, "Visitor Team Rest Days"] = min(3, date_num - visitor_last-1)
    last_games[home] = date_num
    last_games[visitor] = date_num

In [ ]:
modern.to_csv("csvs/modern.csv")

In [ ]:
players_mp = pd.read_csv("csvs/players_game_time.csv")

In [ ]:
player_last_games = {}
for idx, game in players_mp.iterrows():
    player = game["Player ID"]
    date_num = game["Date Number"]
    last = player_last_games.get(player, 0)
    players_mp.loc[idx, "Rest Days"] = min(3, date_num - last-1)
    player_last_games[player] = date_num

In [ ]:
all_players.to_csv("csvs/all_players.csv")

In [ ]:
modern = calculate_l10s(modern)
modern.to_csv("csvs/modern.csv")

In [ ]:
def calculate_sos(df, date_col="Date Number"):
    home_soses = []
    visitor_soses = []
    # Process year by year
    for year, group in df.groupby("Season"):
        team_schedules = {}
        team_wls = {}
        for idx, row in group.iterrows():
            home = row["Home Team"]
            visitor = row["Visitor Team"]
            team_wls[home] = row["Home Team ELO"]
            team_wls[visitor] = row["Visitor Team ELO"]
            home_sch = team_schedules.get(home, [])
            if home_sch:
                home_tot = 0
                for team in home_sch:
                    home_tot += team_wls[team]
                home_sos = home_tot / len(home_sch)
            else:
                home_sos = 0.5
            home_soses.append(home_sos)

            visitor_sch = team_schedules.get(visitor, [])
            if visitor_sch:
                visitor_tot = 0
                for team in visitor_sch:
                    visitor_tot += team_wls[team]
                visitor_sos = visitor_tot / len(visitor_sch)
            else:
                visitor_sos = 0.5
            visitor_soses.append(visitor_sos)
            
            team_schedules.setdefault(home, [])
            team_schedules.setdefault(visitor, [])
            
            team_schedules[home].append(visitor)
            team_schedules[visitor].append(home)

    df["Home Team SOS"] = home_soses
    df["Visitor Team SOS"] = visitor_soses

    return df

In [ ]:
def calculate_l10s(df, date_col="Date Number"):
    home_l10s = []
    visitor_l10s = []
    # Process year by year
    for year, group in df.groupby("Season"):
        team_results = {}
        for idx, row in group.iterrows():
            home = row["Home Team"]
            visitor = row["Visitor Team"]
            home_win = row["Home Win"]
            home_res = team_results.get(home, [])
            if home_res:
                home_l10 = sum(home_res[:10]) / min(10, len(home_res))
            else:
                home_l10 = 0.5
            home_l10s.append(home_l10)

            visitor_res = team_results.get(visitor, [])
            if visitor_res:
                visitor_l10 = sum(visitor_res[:10]) / min(10, len(visitor_res))
            else:
                visitor_l10 = 0.5
            visitor_l10s.append(visitor_l10)
            
            team_results.setdefault(home, [])
            team_results.setdefault(visitor, [])
            
            team_results[home].insert(0, home_win)
            team_results[visitor].insert(0, 1-home_win)

    df["Home Team Last 10 W/L%"] = home_l10s
    df["Visitor Team Last 10 W/L%"] = visitor_l10s

    return df

In [ ]:
def add_possessions(df: pd.DataFrame) -> pd.DataFrame:
    # Home team possessions
    home_team_poss = (
        df["Home Team FGA"]
        + 0.4 * df["Home Team FTA"]
        - 1.07 * (df["Home Team ORB"] / (df["Home Team ORB"] + df["Visitor Team DRB"])) * (df["Home Team FGA"] - df["Home Team FG"])
        + df["Home Team TOV"]
    )

    # Visitor team possessions
    visitor_team_poss = (
        df["Visitor Team FGA"]
        + 0.4 * df["Visitor Team FTA"]
        - 1.07 * (df["Visitor Team ORB"] / (df["Visitor Team ORB"] + df["Home Team DRB"])) * (df["Visitor Team FGA"] - df["Visitor Team FG"])
        + df["Visitor Team TOV"]
    )

    # More specific possession formula (average of both)
    df["Possessions"] = 0.5 * (home_team_poss + visitor_team_poss)

    return df

In [ ]:
modern = calculate_avg_poss(modern, "Possessions")

In [ ]:
def calculate_avg_poss(df, stat_col, date_col="Date Number"):
    home_averages = []
    visitor_averages = []

    # Process year by year
    for year, group in df.groupby("Season"):
        stat_dict = {}
        for idx, row in group.iterrows():
            home_team = row["Home Team"]
            visitor_team = row["Visitor Team"]
            stat = row.get(f"{stat_col}", None)
            
            home_data = stat_dict.get(home_team, {"total": 0, "games": 0})
            visitor_data = stat_dict.get(visitor_team, {"total": 0, "games": 0})
            
            home_avg = (
                home_data["total"] / home_data["games"]
                if home_data["games"] > 0 else None
            )
            visitor_avg = (
                visitor_data["total"] / visitor_data["games"]
                if visitor_data["games"] > 0 else None
            )

            home_averages.append(home_avg)
            visitor_averages.append(visitor_avg)

            # Update totals *after* recording current averages
            stat_dict.setdefault(home_team, {"total": 0, "games": 0})
            stat_dict.setdefault(visitor_team, {"total": 0, "games": 0})

            if pd.notna(stat):
                stat_dict[visitor_team]["total"] += stat
                stat_dict[visitor_team]["games"] += 1
                stat_dict[home_team]["total"] += stat
                stat_dict[home_team]["games"] += 1

    # Add columns back to the original df (in order)
    df[f"Home Team {stat_col} Avg"] = home_averages
    df[f"Visitor Team {stat_col} Avg"] = visitor_averages

    return df

In [ ]:
for col in ["Home Team Possessions Avg", "Visitor Team Possessions Avg"]:
    modern[col] = modern[col].fillna(modern[col].mean())

In [ ]:
modern = modern.rename(columns={"Home Team Possessions Avg": "Home Team Pace Avg", "Visitor Team Possessions Avg": "Visitor Team Pace Avg"})

In [ ]:
def calculate_league_totals(df):
    stats_needed = ["AST","FG","FT","FTA","ORB","TRB","PTS","FGA","TOV","PF"]
    for stat in stats_needed:
        print(stat)
        daily_totals = df.groupby(["Season", "Date Number"])[stat].sum().reset_index()
        
        daily_totals[f"lgCum{stat}"] = daily_totals.groupby("Season")[stat].cumsum()
        daily_totals[f"lgPrevCum{stat}"] = daily_totals.groupby("Season")[f"lgCum{stat}"].shift(fill_value=0)

        # Step 4: merge back so each game gets the right "previous days" total
        df = df.merge(daily_totals[["Season", "Date Number", f"lgPrevCum{stat}"]], 
                      on=["Season", "Date Number"], how="left")

    stats_needed = ["MP","FG","FGA","3P","FT","FTA","AST","ORB","TRB","BLK","STL","TOV","PF","PTS"]
    for stat in stats_needed:
        print("player", stat)
        daily_totals = (
            df.groupby(["Season", "Player ID", "Date Number"])[stat]
              .sum()
              .reset_index()
        )
    
        # Step 2: cumulative and lagged cumulative (per player & season)
        daily_totals[f"Cum{stat}"] = (
            daily_totals.groupby(["Season", "Player ID"])[stat].cumsum()
        )
        daily_totals[f"PrevCum{stat}"] = (
            daily_totals.groupby(["Season", "Player ID"])[f"Cum{stat}"].shift(fill_value=0)
        )
    
        # Step 3: merge back to main df
        df = df.merge(
            daily_totals[["Season", "Player ID", "Date Number", f"PrevCum{stat}"]],
            on=["Season", "Player ID", "Date Number"],
            how="left"
        )

    stats_needed = ["AST","FG"]
    for stat in stats_needed:
        print("team", stat)
        daily_totals = (
            df.groupby(["Season", "Team Name", "Date Number"])[stat]
              .sum()
              .reset_index()
        )
    
        # Step 2: cumulative and lagged cumulative (per player & season)
        daily_totals[f"tmCum{stat}"] = (
            daily_totals.groupby(["Season", "Team Name"])[stat].cumsum()
        )
        daily_totals[f"tmPrevCum{stat}"] = (
            daily_totals.groupby(["Season", "Team Name"])[f"tmCum{stat}"].shift(fill_value=0)
        )
    
        # Step 3: merge back to main df
        df = df.merge(
            daily_totals[["Season", "Team Name", "Date Number", f"tmPrevCum{stat}"]],
            on=["Season", "Team Name", "Date Number"],
            how="left"
        )

    print("league pace")
    lg_daily = (
        df.groupby(["Season", "Date Number"])["Possessions"].mean().reset_index(name="lgAvgPaceDay")
    )
    # cumulative average pace up to previous day
    lg_daily["lgCumPoss"] = lg_daily.groupby("Season")["lgAvgPaceDay"].cumsum()
    lg_daily["lgGames"] = lg_daily.groupby("Season").cumcount() + 1
    lg_daily["lgAvgPace"] = lg_daily["lgCumPoss"] / lg_daily["lgGames"]
    lg_daily["lgPrevAvgPace"] = lg_daily.groupby("Season")["lgAvgPace"].shift(fill_value=np.nan)

    df = df.merge(
        lg_daily[["Season", "Date Number", "lgPrevAvgPace"]],
        on=["Season", "Date Number"],
        how="left"
    )

    # ----------------- Team Pace (Possessions) -----------------
    print("team pace")
    tm_daily = (
        df.groupby(["Season", "Team Name", "Date Number"])["Possessions"].mean().reset_index(name="tmAvgPaceDay")
    )
    tm_daily["tmCumPoss"] = tm_daily.groupby(["Season", "Team Name"])["tmAvgPaceDay"].cumsum()
    tm_daily["tmGames"] = tm_daily.groupby(["Season", "Team Name"]).cumcount() + 1
    tm_daily["tmAvgPace"] = tm_daily["tmCumPoss"] / tm_daily["tmGames"]
    tm_daily["tmPrevAvgPace"] = tm_daily.groupby(["Season", "Team Name"])["tmAvgPace"].shift(fill_value=np.nan)

    df = df.merge(
        tm_daily[["Season", "Team Name", "Date Number", "tmPrevAvgPace"]],
        on=["Season", "Team Name", "Date Number"],
        how="left"
    )
    
    return df



def calculate_uPER(mp, FG, FGA, threeP, FT, FTA, AST, ORB, TRB, BLK, STL, TO, PF,
                   tmAST, tmFG, lgAST, lgFG, lgFT, lgFTA, lgORB, lgTRB, lgPTS, lgFGA, lgTO, lgPF):
    if mp == 0:
        return 0

    # League factors
    factor = (2/3) - ((0.5 * lgAST / lgFG) / (2 * (lgFG / lgFT)))
    VOP = lgPTS / (lgFGA - lgORB + lgTO + 0.44 * lgFTA)
    DRBP = (lgTRB - lgORB) / lgTRB

    uPER = (1/mp) * (
        threeP
        - (PF * lgFT / lgPF)
        + (FT / 2) * (2 - (tmAST / (3 * tmFG)))
        + FG * (2 - (factor * tmAST / tmFG))
        + (2 * AST / 3)
        + VOP * (
            DRBP * (2 * ORB + BLK - 0.2464 * (FTA - FT) - (FGA - FG) - TRB)
            + (0.44 * lgFTA * PF / lgPF)
            - (TO + ORB)
            + STL
            + TRB
            - 0.1936 * (FTA - FT)
        )
    )
    return uPER

# -------------------------
# Rolling PER Calculator
# -------------------------
def calculate_rolling_PER(df):
    """
    Calculate rolling PER for each player up to (but excluding) each game.
    Requires df with: Season, Date Number, Player, Team, and box score stats.
    """
    df["uPER"] = df.apply(
        lambda row: calculate_uPER(
            row["PrevCumMP"],
            row["PrevCumFG"], row["PrevCumFGA"], row["PrevCum3P"],
            row["PrevCumFT"], row["PrevCumFTA"], row["PrevCumAST"],
            row["PrevCumORB"], row["PrevCumTRB"], row["PrevCumBLK"],
            row["PrevCumSTL"], row["PrevCumTOV"], row["PrevCumPF"],
    
            row["tmPrevCumAST"], row["tmPrevCumFG"],  # team-level
            row["lgPrevCumAST"], row["lgPrevCumFG"], row["lgPrevCumFT"],
            row["lgPrevCumFTA"], row["lgPrevCumORB"], row["lgPrevCumTRB"],
            row["lgPrevCumPTS"], row["lgPrevCumFGA"], row["lgPrevCumTOV"],
            row["lgPrevCumPF"],  # league-level
        ),
        axis=1
    )
    
    lg_daily = (
        df.groupby(["Season", "Date Number"])["uPER"].mean().reset_index(name="lguPER")
    )
    # cumulative average pace up to previous day
    lg_daily["lgCumuPER"] = lg_daily.groupby("Season")["lguPER"].cumsum()
    lg_daily["lgGames"] = lg_daily.groupby("Season").cumcount() + 1
    lg_daily["lgAvguPER"] = lg_daily["lgCumuPER"] / lg_daily["lgGames"]

    df = df.merge(
        lg_daily[["Season", "Date Number", "lgAvguPER"]],
        on=["Season", "Date Number"],
        how="left"
    )
    for col in ["uPER", "lgPrevAvgPace", "tmPrevAvgPace", "lgAvguPER"]:
        df[col] = df[col].fillna(df[col].mean())
    df["PER"] = df["uPER"] * (df["lgPrevAvgPace"] / df["tmPrevAvgPace"]) * (15 / df["lgAvguPER"])
    return df


In [ ]:
with_per = calculate_rolling_PER(with_totals)

In [ ]:
modern_players = players_gt[players_gt["Season"] >= 1986]
modern_players = modern_players.merge(
    modern[["Game ID", "Possessions"]],
    on="Game ID",
    how="left"
)
with_totals = calculate_league_totals(modern_players)

In [650]:
with_per.to_csv("csvs/with_per.csv")

In [ ]:
print(with_per[with_per["Season"] == 2019]["PER"].mean())

In [ ]:
for col in ["PER"]:
    with_per[col] = with_per[col].fillna(15)

In [ ]:
with_per["PER"].isna().sum()

Unnamed: 0        0
Player ID         0
Game ID           0
Player Name       0
Team Name         0
Opp Team Name     0
Date              0
Date Number       0
Season            0
Game Type         0
Win               0
Home              0
MP                0
PTS               0
AST               0
BLK               0
STL               0
FGA               0
FG                0
FG%               0
3PA               0
3P                0
3P%               0
FTA               0
FT                0
FT%               0
DRB               0
ORB               0
TRB               0
PF                0
TOV               0
+/-               0
EFF               0
Season EFF Avg    0
Career EFF Avg    0
Possessions       0
PER               0
MP Avg            0
dtype: int64

In [685]:
def calculate_team_pers(df):
    season_pers = []
    for idx, row in df.iterrows():
        home = row["Home Team"]
        visitor = row["Visitor Team"]
        game_id = row["Game ID"]
        game_players = modern_players[modern_players["Game ID"] == game_id]
        home_players = game_players[game_players["Team Name"] == home]
        visitor_players = game_players[game_players["Team Name"] == visitor]
        home_min = home_players["MP Avg"].sum()
        visitor_min = visitor_players["MP Avg"].sum()
        home_tot = (home_players["PER"] * home_players["MP Avg"]).sum()
        visitor_tot = (visitor_players["PER"] * visitor_players["MP Avg"]).sum()
        df.loc[idx, "Home Team Standardized PER"] = home_tot * 240 / home_min if home_min > 0 else 0
        df.loc[idx, "Visitor Team Standardized PER"] = visitor_tot * 240 / visitor_min if visitor_min > 0 else 0

    return df

In [686]:
modern = calculate_team_pers(modern)

In [687]:
modern.to_csv("csvs/modern.csv")

In [658]:
def calculate_avg_mp(df):
    # Step 1: aggregate to daily level
    player_daily = (
        df.groupby(["Season", "Player ID", "Date Number"])["MP"]
          .sum()
          .reset_index(name="MP_day")
    )
    
    # Step 2: rolling average of previous games (exclude current row with shift)
    player_daily["MP Avg"] = (
        player_daily.groupby(["Season", "Player ID"])["MP_day"]
        .transform(lambda s: s.shift(1).rolling(window=100, min_periods=1).mean())
    )
    
    # Step 3: merge back
    df = df.merge(
        player_daily[["Season", "Player ID", "Date Number", "MP Avg"]],
        on=["Season", "Player ID", "Date Number"],
        how="left"
    )
    
    return df


In [661]:
modern_players = players_gt[players_gt["Season"] >= 1986]
modern_players = modern_players.merge(
    modern[["Game ID", "Possessions"]],
    on="Game ID",
    how="left"
)
modern_players["PER"] = with_per["PER"].copy()

In [662]:
modern_players = calculate_avg_mp(modern_players)

In [665]:
modern_players.to_csv("csvs/modern_players.csv")

In [664]:
modern_players["MP Avg"] = modern_players["MP Avg"].fillna(modern_players["MP Avg"].mean())

In [691]:
modern["Home Team TS%"] = 0.5 * modern["Home Team PTS"] / (modern["Home Team FGA"] + 0.44 * modern["Home Team FTA"])
modern["Visitor Team TS%"] = 0.5 * modern["Visitor Team PTS"] / (modern["Visitor Team FGA"] + 0.44 * modern["Visitor Team FTA"])

modern["Home Team TOV%"] = modern["Home Team TOV"] / (modern["Home Team FGA"] + 0.44 * modern["Home Team FTA"] + modern["Home Team TOV"])
modern["Visitor Team TOV%"] = modern["Visitor Team TOV"] / (modern["Visitor Team FGA"] + 0.44 * modern["Visitor Team FTA"]+ modern["Visitor Team TOV"])

modern["Home Team ORB%"] = modern["Home Team ORB"] / (modern["Home Team ORB"] + modern["Visitor Team DRB"])
modern["Visitor Team ORB%"] = modern["Visitor Team ORB"] / (modern["Visitor Team ORB"] + modern["Home Team DRB"])

modern["Home Team FT/FGA"] = modern["Home Team FT"] / modern["Home Team FGA"]
modern["Visitor Team FT/FGA"] = modern["Visitor Team FT"] / modern["Visitor Team FGA"]

modern["Home Team TS% Avg"] = 0.5 * modern["Home Team PTS Avg"] / (modern["Home Team FGA Avg"] + 0.44 * modern["Home Team FTA Avg"])
modern["Visitor Team TS% Avg"] = 0.5 * modern["Visitor Team PTS Avg"] / (modern["Visitor Team FGA Avg"] + 0.44 * modern["Visitor Team FTA Avg"])

modern["Home Team TOV% Avg"] = modern["Home Team TOV Avg"] / (modern["Home Team FGA Avg"] + 0.44 * modern["Home Team FTA Avg"] + modern["Home Team TOV Avg"])
modern["Visitor Team TOV% Avg"] = modern["Visitor Team TOV Avg"] / (modern["Visitor Team FGA Avg"] + 0.44 * modern["Visitor Team FTA Avg"]+ modern["Visitor Team TOV Avg"])

modern["Home Team ORB% Avg"] = modern["Home Team ORB Avg"] / (modern["Home Team ORB Avg"] + modern["Visitor Team DRB Avg"])
modern["Visitor Team ORB% Avg"] = modern["Visitor Team ORB Avg"] / (modern["Visitor Team ORB Avg"] + modern["Home Team DRB Avg"])

modern["Home Team FT/FGA Avg"] = modern["Home Team FT Avg"] / modern["Home Team FGA Avg"]
modern["Visitor Team FT/FGA Avg"] = modern["Visitor Team FT Avg"] / modern["Visitor Team FGA Avg"]

In [694]:
modern.to_csv('csvs/modern.csv')

In [695]:
text = """
Abbreviation,Name
ATL,Atlanta Hawks
BRK,Brooklyn Nets
BKN,Brooklyn Nets
BOS,Boston Celtics
CHA,Charlotte Bobcats
CHH,Charlotte Hornets
CHO,Charlotte Hornets
CHI,Chicago Bulls
CLE,Cleveland Cavaliers
DAL,Dallas Mavericks
DEN,Denver Nuggets
DET,Detroit Pistons
GSW,Golden State Warriors
HOU,Houston Rockets
IND,Indiana Pacers
LAC,Los Angeles Clippers
LAL,Los Angeles Lakers
MEM,Memphis Grizzlies
MIA,Miami Heat
MIL,Milwaukee Bucks
MIN,Minnesota Timberwolves
NJN,New Jersey Nets
NOH,New Orleans Hornets
NOP,New Orleans Pelicans
NOK,Oklahoma City Hornets
NYK,New York Knicks
OKC,Oklahoma City Thunder
ORL,Orlando Magic
PHI,Philadelphia 76ers
PHX,Phoenix Suns
PHO,Phoenix Suns
POR,Portland Trail Blazers
SEA,Seattle SuperSonics
SAC,Sacramento Kings
SAS,San Antonio Spurs
TOR,Toronto Raptors
UTA,Utah Jazz
VAN,Vancouver Grizzlies
WAS,Washington Wizards
WSB,Washington Bullets
"""
df = pd.read_csv(io.StringIO(text))
df.to_csv("csvs/abbrevs.csv")

In [697]:
print(len(df))

40


In [698]:
print(len(modern["Home Team"].unique()))

37


In [700]:
print(sorted(modern["Home Team"].unique()))

['Atlanta Hawks', 'Boston Celtics', 'Brooklyn Nets', 'Charlotte Bobcats', 'Charlotte Hornets', 'Chicago Bulls', 'Cleveland Cavaliers', 'Dallas Mavericks', 'Denver Nuggets', 'Detroit Pistons', 'Golden State Warriors', 'Houston Rockets', 'Indiana Pacers', 'Los Angeles Clippers', 'Los Angeles Lakers', 'Memphis Grizzlies', 'Miami Heat', 'Milwaukee Bucks', 'Minnesota Timberwolves', 'New Jersey Nets', 'New Orleans Hornets', 'New Orleans Pelicans', 'New York Knicks', 'Oklahoma City Hornets', 'Oklahoma City Thunder', 'Orlando Magic', 'Philadelphia 76ers', 'Phoenix Suns', 'Portland Trail Blazers', 'Sacramento Kings', 'San Antonio Spurs', 'Seattle SuperSonics', 'Toronto Raptors', 'Utah Jazz', 'Vancouver Grizzlies', 'Washington Bullets', 'Washington Wizards']


In [2]:
base_url = "https://www.basketball-reference.com/leagues/NBA_2026_games-{}.html"
months = ["october", "november", "december", "january", "february", "march", "april"]

from bs4 import BeautifulSoup
import requests

In [3]:

for month in months:
    url = base_url.format(month)
    data = requests.get(url)
    
    with open("schedules/2026{}.html".format(month), "w+") as f:
        f.write(data.text)

In [58]:
from io import StringIO

all_games = []
for month in months:
    with open("schedules/2026{}.html".format(month)) as f:
        page = f.read()
    soup = BeautifulSoup(page,"html.parser")
    game_table = soup.find(id="schedule")
    game = pd.read_html(StringIO(str(game_table)))[0]
    game["Season"] = 2026
    game["Game Type"] = 'Regular Season'
    game["Old Date"] = game["Date"]
    all_games.append(game)
games = pd.concat(all_games)
games.to_csv("csvs/games_2026.csv")

In [710]:
!pip install beautifulsoup4

In [712]:
!pip install html5lib lxml

In [60]:
def change_date_form(date, time):
    month_to_num = {'Jan': 1, 'Feb': 2, 'Mar': 3, 'Apr': 4, 'May': 5, 'Jun': 6, 'Jul': 7,
                    'Aug': 8, 'Sep': 9, 'Oct': 10, 'Nov': 11, 'Dec': 12}
    date_split = date.split(', ')
    month_date = date_split[1]
    year = date_split[2]
    month_split = month_date.split(' ')
    month = month_split[0]
    day_of_month = month_split[1]
    if not time:
        time = "12:00a"
    if isinstance(time, float):
        time = "12:00a"
    time_split = time.split(':')
    hour = int(time_split[0])
    if 'p' in time_split[-1]:
        hour = hour + 12 if hour != 12 else 12
    if 'a' in time_split[-1]:
        hour = hour - 12 if hour == 12 else hour
    time_split[1]
    return f"{year}-{month_to_num[month]}-{day_of_month} {hour}:{time_split[1][:-1]}:00"

    
    

In [34]:
print(change_date_form("Tue, Oct 21, 2025", "11:00p"))

['Tue', 'Oct 21', '2025']
['Oct', '21']
['11', '00p']
11
2025-10-21 23:00:00


In [63]:
games.to_csv("csvs/games_2026.csv")

In [49]:
games = games.rename(columns={"Date": "Old Date"})

In [ ]:
games["Date"] = games.apply(lambda row: change_date_form(row["Old Date"], row["Start (ET)"]), axis=1)

In [52]:
games["Game Type"] = "Regular Season"

In [62]:
games = games.rename(columns={"Home/Neutral": "Home Team", "Visitor/Neutral": "Visitor Team"})

In [64]:
games.columns

Index(['Date', 'Start (ET)', 'Visitor Team', 'PTS', 'Home Team', 'PTS.1',
       'Unnamed: 6', 'Unnamed: 7', 'Attend.', 'LOG', 'Arena', 'Notes',
       'Season', 'Game Type', 'Old Date'],
      dtype='object')

In [65]:
games_useful = games[['Date', 'Visitor Team', 'Home Team',
       'Season', 'Game Type']]

In [67]:
modern = pd.read_csv("csvs/modern.csv")

In [71]:
modern_26 = pd.concat([modern, games_useful], ignore_index=True)
modern_26 = modern_26.loc[:, ~modern_26.columns.str.contains("^Unnamed")]
modern_26.to_csv("csvs/modern26.csv")

In [70]:
modern.to_csv("csvs/modern.csv")

In [74]:
modern_26["Date Number"] = modern_26.apply(lambda row: date_to_num(row["Date"]), axis=1)

In [75]:
modern_26 = modern_26.sort_values(by="Date Number").reset_index(drop=True)

In [76]:
modern_26["Game ID"] = range(20905,71286)

In [77]:
modern_26.to_csv("csvs/modern26.csv")